# Unit commitment with `milp`, knapsack, two-stage scenarios and heuristics

Mixed-integer and scenario methods with nothing but `scipy.optimize` and NumPy: on/off
decisions for generators, which projects to build under a budget, how much to hedge when
the future is a set of scenarios, and what to do when exact optimisation is too slow.

**What's in here**
- `milp` anatomy: cost vector, `LinearConstraint`, `Bounds`, `integrality`, `res.status`
- Variable indexing helper for (generator, hour) → column, checked on a 2×2 toy
- Unit commitment: min/max output, demand balance, start-up costs, minimum up-time
- LP relaxation vs MILP: the integrality gap, and why rounding the LP fails
- Knapsack: which assets to build under a capex budget
- Two-stage stochastic LP: hedge volume now, imbalance recourse per scenario
- Convergence in the number of scenarios and out-of-sample scenario evaluation
- Heuristics: greedy dispatch, local search, `differential_evolution`
- Timing as the horizon grows; formulation tips (big-M, tight bounds)

In [1]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import milp, linprog, LinearConstraint, Bounds, differential_evolution
import matplotlib.pyplot as plt
%matplotlib inline

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
day = df[df["time"].dt.date == pd.Timestamp("2023-01-17").date()]
demand = day["consumption_mwh"].values / 1000            # GW, one cold winter day
print("demand GW: min", demand.min().round(1), "max", demand.max().round(1))

demand GW: min 28.6 max 38.9


## 1. `milp` anatomy

`milp(c, constraints=LinearConstraint(A, lb, ub), integrality=..., bounds=Bounds(lo, hi))`
minimises `c @ x`. Everything is a matrix: you build `A` yourself. `integrality` is a
float array (1 = integer, 0 = continuous). Variable bounds go in `Bounds`, **not** in `A`.
A tiny LP first, to see the object that comes back.

In [2]:
# minimise 3x + 2y  s.t.  x + y >= 4,  x <= 3,  y integer
c = np.array([3.0, 2.0])
cons = LinearConstraint(np.array([[1.0, 1.0]]), lb=4, ub=np.inf)
res = milp(c, constraints=cons, integrality=np.array([0, 1]), bounds=Bounds([0, 0], [3, np.inf]))
print("status:", res.status, "|", res.message)
print("x:", res.x, " objective:", res.fun)

status: 0 | Optimization terminated successfully. (HiGHS Status 7: Optimal)
x: [0. 4.]  objective: 8.0


**Pitfall:** `res.status == 0` is success; 1 = iteration/time limit, 2 = infeasible,
3 = unbounded, 4 = other. Check it every time — an infeasible problem still returns an
object, with `res.x = None`.

## 2. Unit commitment: the variables

Three thermal units `g`, 24 hours `h`. Per (g, h):
- `p[g,h]` output (GW, continuous),
- `u[g,h]` on/off (binary),
- `s[g,h]` start-up indicator (continuous in [0,1]; forced to 1 when the unit turns on).

Put them in one long vector. Write an **index helper** and test it on a toy instance before
trusting it on the real one — indexing bugs are the number one MILP failure.

In [3]:
units = pd.DataFrame({
    "name":     ["nuclear", "ccgt", "peaker"],
    "p_max":    [20.0, 12.0, 10.0],          # GW
    "p_min":    [ 8.0,  4.0,  2.0],          # GW when on
    "mc":       [40.0, 70.0, 120.0],         # €/MWh marginal cost  -> k€ per GWh
    "no_load":  [300.0, 150.0, 50.0],        # k€ per hour when on
    "start_up": [500.0, 200.0, 50.0],        # k€ per start
    "min_up":   [24, 4, 1],                  # hours
}).set_index("name")
units

,p_max,p_min,mc,no_load,start_up,min_up
name,,,,,,
nuclear,20.0,8.0,40.0,300.0,500.0,24
ccgt,12.0,4.0,70.0,150.0,200.0,4
peaker,10.0,2.0,120.0,50.0,50.0,1


In [4]:
class UCIndex:
    def __init__(self, G, H):
        self.G, self.H = G, H
        self.n = 3 * G * H
    def p(self, g, h): return g * self.H + h
    def u(self, g, h): return self.G * self.H + g * self.H + h
    def s(self, g, h): return 2 * self.G * self.H + g * self.H + h

ix = UCIndex(2, 2)
print("toy 2x2: n =", ix.n, "| p(1,1) =", ix.p(1, 1), "u(0,0) =", ix.u(0, 0), "s(1,0) =", ix.s(1, 0))
cols = sorted([ix.p(g, h) for g in range(2) for h in range(2)] + [ix.u(g, h) for g in range(2) for h in range(2)]
              + [ix.s(g, h) for g in range(2) for h in range(2)])
assert cols == list(range(ix.n)), "index helper does not tile the vector exactly"
print("index helper covers every column exactly once")

toy 2x2: n = 12 | p(1,1) = 3 u(0,0) = 4 s(1,0) = 10
index helper covers every column exactly once


## 3. Constraints as rows of `A`

For each hour: `Σ_g p[g,h] = demand[h]`.
For each (g, h): `p ≤ p_max · u`, `p ≥ p_min · u`, `s ≥ u[h] − u[h−1]`.
Minimum up-time for unit g with `min_up = k`: if it starts at `h`, it stays on for hours
`h..h+k−1`: `u[g, h'] ≥ s[g, h]` for `h' = h..h+k−1`.

Build rows as dense NumPy arrays (fine at this size), with `lb`/`ub` per row; equality is
`lb == ub`.

In [5]:
def build_uc(demand, units, min_up=True, u_init=None):
    G, H = len(units), len(demand)
    ix = UCIndex(G, H)
    c = np.zeros(ix.n)
    for g, (_, r) in enumerate(units.iterrows()):
        for h in range(H):
            c[ix.p(g, h)] = r.mc            # €/MWh == k€/GWh
            c[ix.u(g, h)] = r.no_load
            c[ix.s(g, h)] = r.start_up
    rows, lb, ub = [], [], []
    def add(row, lo, hi): rows.append(row); lb.append(lo); ub.append(hi)

    for h in range(H):                                   # demand balance
        a = np.zeros(ix.n)
        for g in range(G): a[ix.p(g, h)] = 1
        add(a, demand[h], demand[h])
    for g, (_, r) in enumerate(units.iterrows()):
        for h in range(H):
            a = np.zeros(ix.n); a[ix.p(g, h)] = 1; a[ix.u(g, h)] = -r.p_max; add(a, -np.inf, 0)   # p <= pmax u
            a = np.zeros(ix.n); a[ix.p(g, h)] = 1; a[ix.u(g, h)] = -r.p_min; add(a, 0, np.inf)    # p >= pmin u
            a = np.zeros(ix.n); a[ix.s(g, h)] = 1; a[ix.u(g, h)] = -1                              # s >= u_h - u_{h-1}
            if h > 0: a[ix.u(g, h - 1)] = 1; add(a, 0, np.inf)
            else: add(a, -(u_init[g] if u_init is not None else 0), np.inf)
            if min_up:
                for k in range(1, int(r.min_up)):
                    if h + k < H:
                        a = np.zeros(ix.n); a[ix.u(g, h + k)] = 1; a[ix.s(g, h)] = -1; add(a, 0, np.inf)   # u_{h+k} >= s_h
    A = np.array(rows)
    integrality = np.zeros(ix.n); integrality[G * H: 2 * G * H] = 1
    hi = np.full(ix.n, np.inf); hi[G * H:] = 1.0
    return c, LinearConstraint(A, lb, ub), integrality, Bounds(0, hi), ix

c, cons, integ, bnds, ix = build_uc(demand, units)
print("variables:", ix.n, " constraint rows:", cons.A.shape[0])

variables: 216  constraint rows: 582


In [6]:
t0 = time.time()
res = milp(c, constraints=cons, integrality=integ, bounds=bnds)
print(f"status {res.status} | {res.message} | {time.time()-t0:.2f}s")
print(f"total cost: {res.fun:,.0f} k€")

def unpack(res, ix, units):
    G, H = ix.G, ix.H
    x = res.x
    p = pd.DataFrame({u: x[[ix.p(g, h) for h in range(H)]] for g, u in enumerate(units.index)})
    on = pd.DataFrame({u: x[[ix.u(g, h) for h in range(H)]] for g, u in enumerate(units.index)}).round().astype(int)
    st = pd.DataFrame({u: x[[ix.s(g, h) for h in range(H)]] for g, u in enumerate(units.index)}).round().astype(int)
    return p, on, st

p, on, st = unpack(res, ix, units)
print("starts per unit:", st.sum().to_dict())
on.T

status 0 | Optimization terminated successfully. (HiGHS Status 7: Optimal) | 0.00s
total cost: 56,604 k€
starts per unit: {'nuclear': 1, 'ccgt': 1, 'peaker': 1}


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
nuclear,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
ccgt,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
peaker,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0


The peaker is switched on only for the daytime block, and only once (one start-up cost).
Always eyeball the schedule: does the dispatch meet demand every hour, do units respect
`p_min` when on and produce zero when off?

In [7]:
checks = pd.Series({
    "demand met (max abs gap GW)": np.abs(p.sum(axis=1).values - demand).max(),
    "output when off (max GW)":    (p.values * (1 - on.values)).max(),
    "below p_min when on (count)": int(((p.values < units.p_min.values - 1e-6) & (on.values == 1)).sum()),
    "above p_max (count)":         int((p.values > units.p_max.values + 1e-6).sum()),
})
checks.round(6)

demand met (max abs gap GW)    0.0
output when off (max GW)       0.0
below p_min when on (count)    0.0
above p_max (count)            0.0
dtype: float64

## 4. LP relaxation and the integrality gap

Drop `integrality` and the same problem is an LP: `u` becomes a fraction. The LP objective
is a **lower bound** on the MILP; the difference is the integrality gap. Rounding the LP's
fractional `u` does not give a feasible schedule.

In [8]:
res_lp = milp(c, constraints=cons, integrality=np.zeros(ix.n), bounds=bnds)
p_lp, on_lp_raw, _ = unpack(res_lp, ix, units)
u_lp = pd.DataFrame({u: res_lp.x[[ix.u(g, h) for h in range(ix.H)]] for g, u in enumerate(units.index)})
print(f"MILP cost {res.fun:,.0f} k€ | LP relaxation {res_lp.fun:,.0f} k€ | gap {res.fun/res_lp.fun-1:.2%}")
print("fractional on/off values in the LP (peaker, first 12 hours):", u_lp["peaker"].values[:12].round(2))

MILP cost 56,604 k€ | LP relaxation 55,871 k€ | gap 1.31%
fractional on/off values in the LP (peaker, first 12 hours): [0.13 0.   0.   0.   0.   0.   0.   0.21 0.4  0.5  0.45 0.38]


In [9]:
# 'Round the LP' heuristic: fix u to round(u_lp) and re-solve for p only. Is it even feasible?
u_round = (u_lp.values > 0.5).astype(float)
G, H = ix.G, ix.H
lo = np.zeros(ix.n); hi = np.full(ix.n, np.inf); hi[G*H:] = 1.0
for g in range(G):
    for h in range(H):
        lo[ix.u(g, h)] = hi[ix.u(g, h)] = u_round[h, g]
res_round = milp(c, constraints=cons, integrality=np.zeros(ix.n), bounds=Bounds(lo, hi))
print("rounded-LP status:", res_round.status, "|", res_round.message)
if res_round.status == 0:
    print(f"rounded-LP cost {res_round.fun:,.0f} k€ vs MILP {res.fun:,.0f} k€  ({res_round.fun/res.fun-1:+.2%})")

rounded-LP status: 2 | The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is At lower/fixed bound)


**Interview check:** "The LP is 1–2% cheaper. Why not use it?" — Because a unit that is
"37% on" does not exist. The LP value is a bound, useful to judge how good the integer
solution is, not a plan.

**Pitfall:** big-M constraints (`p ≤ M·u`) with `M` far larger than needed make the LP
relaxation weak and the solver slow. Use the actual `p_max`, never `1e6`.

## 5. Which assets to build: a knapsack

Eight candidate projects with capex and expected annual margin; a capex budget. Maximise
margin: `milp` with all-binary variables (minimise the negative margin).

In [10]:
proj = pd.DataFrame({
    "project":  ["wind_A", "wind_B", "solar_A", "solar_B", "battery_1", "battery_2", "ccgt", "solar_C"],
    "capex_m":  [108, 80, 60, 45, 30, 55, 200, 25],
    "margin_m": [18, 11, 7, 3, 5, 9, 26, 3],
}).set_index("project")
proj["margin_per_capex"] = (proj.margin_m / proj.capex_m).round(3)
budget = 250

kn = milp(-proj.margin_m.values,
          constraints=LinearConstraint(proj.capex_m.values[None, :], -np.inf, budget),
          integrality=np.ones(len(proj)), bounds=Bounds(0, 1))
chosen = proj.index[kn.x > 0.5].tolist()
print(f"status {kn.status} | build {chosen} | capex {proj.loc[chosen, 'capex_m'].sum()} / {budget} | margin {-kn.fun:.0f} M€/yr")

# greedy by margin/capex ratio, for comparison
greedy, spent = [], 0
for name, r in proj.sort_values("margin_per_capex", ascending=False).iterrows():
    if spent + r.capex_m <= budget: greedy.append(name); spent += r.capex_m
print(f"greedy by ratio: {greedy} | capex {spent} | margin {proj.loc[greedy, 'margin_m'].sum():.0f} M€/yr")
kn_lp = milp(-proj.margin_m.values, constraints=LinearConstraint(proj.capex_m.values[None, :], -np.inf, budget),
             integrality=np.zeros(len(proj)), bounds=Bounds(0, 1))
print(f"LP relaxation: {-kn_lp.fun:.2f} M€/yr with fractional x = {kn_lp.x.round(2)}")

status 0 | build ['wind_A', 'wind_B', 'battery_2'] | capex 243 / 250 | margin 38 M€/yr
greedy by ratio: ['wind_A', 'battery_1', 'battery_2', 'solar_C'] | capex 218.0 | margin 35 M€/yr
LP relaxation: 39.84 M€/yr with fractional x = [1.   0.71 0.   0.   1.   1.   0.   0.  ]


The greedy ratio rule fills up with small high-ratio projects and then cannot fit anything
else, leaving 32 M€ of budget unused and 3 M€/yr of margin on the table; the exact answer
swaps battery_1 and solar_C for wind_B. Greedy is a fine first guess and a sanity check; the
MILP is cheap at this size, so use it. The LP relaxation builds "71% of wind_B", the usual
reminder that its value is a bound, not a plan.

## 6. Two-stage stochastic hedging

Decide **now** a flat forward volume `F` (MWh per hour for a month) at fixed price `K`.
Later, in each scenario `s` (load `L_s`, spot `P_s`), the imbalance settles: shortfall
bought at `P_s + c_u`, surplus sold at `P_s − c_o`. Minimise expected cost:

`min_F  K·F + (1/S) Σ_s [ (P_s + c_u)·short_s − (P_s − c_o)·long_s ]`
`s.t.   F + short_s − long_s = L_s,  short_s, long_s ≥ 0`

One first-stage variable, `2S` recourse variables, `S` equality rows. Scenarios are
bootstrapped from the data with multiplicative noise.

**Pitfall:** the data contains negative prices. A scenario with `P_s + c_u < 0` gives the
`short_s` variable a *negative* cost, so the LP happily buys an infinite shortfall and
returns `status 3` (unbounded). The first version of this cell did exactly that. Either
clip scenario prices or model the imbalance price floor explicitly — and always check the
status instead of reading `res.x`.

A second, subtler unboundedness: with few scenarios the sample average of `P_s − c_o` can
exceed the forward price `K`, and then buying an infinite forward volume and selling it all
back is "free money". Real desks have position limits; give `F` an upper bound and check
whether the solution sits on it.

In [11]:
jan = df[(df["time"] >= "2023-01-01") & (df["time"] < "2023-02-01")]
L_hist = jan["consumption_mwh"].values * 0.05           # a 5% retailer
P_hist = jan["price_eur_mwh"].values
K = P_hist.mean()                                       # forward price = expected spot (no risk premium)
c_u, c_o = 25.0, 12.0

def make_scenarios(S, seed):
    r = np.random.default_rng(seed)
    idx = r.integers(0, len(L_hist), S)
    L = L_hist[idx] * r.lognormal(0, 0.05, S)
    P = P_hist[idx] * r.lognormal(0, 0.15, S)
    P = np.clip(P, 0, None)          # see the pitfall below: P + c_u < 0 makes the LP unbounded
    return L, P

def solve_hedge(L, P):
    S = len(L)
    n = 1 + 2 * S
    c = np.zeros(n); c[0] = K
    c[1:1 + S] = (P + c_u) / S
    c[1 + S:] = -(P - c_o) / S
    A = np.zeros((S, n)); A[:, 0] = 1
    A[np.arange(S), 1 + np.arange(S)] = 1
    A[np.arange(S), 1 + S + np.arange(S)] = -1
    F_max = 1.5 * L_hist.max()                              # position limit: see pitfall above
    res = linprog(c, A_eq=A, b_eq=L, bounds=[(0, F_max)] + [(0, None)] * (2 * S), method="highs")
    assert res.status == 0, res.message
    return res.x[0], res.fun

L, P = make_scenarios(200, 1)
F_star, exp_cost = solve_hedge(L, P)
print(f"forward volume F* = {F_star:,.0f} MWh/h | expected hourly cost {exp_cost:,.0f} € | on position limit: {np.isclose(F_star, 1.5 * L_hist.max())}")
print(f"mean load {L_hist.mean():,.0f} | load quantile at c_u/(c_u+c_o) = {np.quantile(L_hist, c_u/(c_u+c_o)):,.0f}")

forward volume F* = 1,728 MWh/h | expected hourly cost 196,492 € | on position limit: False
mean load 1,617 | load quantile at c_u/(c_u+c_o) = 1,718


With `K = E[P]` and independent price noise, the two-stage LP rediscovers the newsvendor
quantile: hedge the load quantile at `c_u / (c_u + c_o)`, not the mean. When price and load
are correlated (cold → high load *and* high price) the LP hedges more; that is the case the
formula does not cover and the scenario method does.

### Convergence in the number of scenarios

A hedge optimised on 10 scenarios fits those 10 scenarios. Check (a) how `F*` moves with
`S`, and (b) the **out-of-sample** cost on a large independent scenario set.

In [12]:
L_oos, P_oos = make_scenarios(20_000, 999)

def evaluate(F, L, P):
    short = np.clip(L - F, 0, None); long = np.clip(F - L, 0, None)
    return np.mean(K * F + (P + c_u) * short - (P - c_o) * long)

rows = []
for S in [10, 25, 50, 100, 200, 500, 2000]:
    Fs = [solve_hedge(*make_scenarios(S, seed))[0] for seed in range(5)]
    rows.append([S, np.mean(Fs), np.std(Fs), evaluate(np.mean(Fs), L_oos, P_oos)])
conv = pd.DataFrame(rows, columns=["scenarios", "F* mean over 5 draws", "F* std over 5 draws", "OOS expected cost"])
conv["OOS cost vs best"] = conv["OOS expected cost"] - conv["OOS expected cost"].min()
conv.round(1)

,scenarios,F* mean over 5 draws,F* std over 5 draws,OOS expected cost,OOS cost vs best
0,10,1857.8,84.8,197814.5,392.3
1,25,1810.8,133.2,197569.5,147.3
2,50,2041.1,498.1,199393.5,1971.3
3,100,1750.5,83.7,197424.9,2.7
4,200,1746.7,17.9,197423.1,0.9
5,500,1734.8,12.9,197423.5,1.3
6,2000,1741.1,8.9,197422.2,0.0


The standard deviation of `F*` across re-draws is the practical convergence test: when it
stops shrinking relative to the decision's sensitivity, you have enough scenarios. Two
things to notice: at `S = 50` one of the five draws ran into the position limit (the
sample-average arbitrage from the pitfall above), which is why the mean and the standard
deviation jump — convergence is not monotonic with small samples. And how flat the OOS cost
is once `S ≥ 100`: the cost is not very sensitive to `F` near the optimum, which is
comforting for a hedger and a reason not to over-engineer the scenario count.

**Pitfall:** evaluating the hedge on the same scenarios used to optimise it (in-sample) makes
small `S` look best. Always evaluate on a fresh set.

**Interview check:** "How would you add a risk constraint?" — Add a CVaR term: auxiliary
variables `η` and `z_s ≥ cost_s − η`, minimise `E[cost] + λ·(η + (1/(1−α)) · mean(z_s))`.
Still an LP.

## 7. Heuristics when exact is too slow

Three tools, in the order you should reach for them:
1. **Greedy / merit order**: dispatch cheapest units first. Fast, usually close, ignores
   start-ups and min up-time.
2. **Local search**: start from greedy, flip one `u[g,h]`, keep it if cost falls and the
   schedule stays feasible.
3. **`differential_evolution`**: gradient-free global search for small, ugly, continuous
   problems (non-convex, noisy). Never the first choice for something `milp` can solve.

In [13]:
def merit_order_dispatch(demand, units):
    order = units.sort_values("mc")
    p = np.zeros((len(demand), len(units)))
    for h, d in enumerate(demand):
        remaining, prev = d, None
        for g_name, r in order.iterrows():
            g = units.index.get_loc(g_name)
            if remaining <= 1e-9: break
            take = min(r.p_max, remaining)
            if take < r.p_min:                          # cannot run below p_min: run at p_min and back off the cheaper unit
                back_off = r.p_min - take
                if prev is not None and p[h, prev] - back_off >= units.p_min.iloc[prev]:
                    p[h, prev] -= back_off
                    take = r.p_min
            p[h, g] = take; remaining -= take; prev = g
    return p

def schedule_cost(p, units):
    on = (p > 1e-9).astype(float)
    starts = np.clip(np.diff(on, axis=0, prepend=0), 0, None)
    return (p * units.mc.values).sum() + (on * units.no_load.values).sum() + (starts * units.start_up.values).sum()

p_greedy = merit_order_dispatch(demand, units)
print(f"greedy merit-order cost: {schedule_cost(p_greedy, units):,.0f} k€  (supply-demand max gap {np.abs(p_greedy.sum(1)-demand).max():.2f} GW)")
print(f"MILP optimum:            {res.fun:,.0f} k€")
print("greedy peaker on/off:", (p_greedy[:, 2] > 0).astype(int))

greedy merit-order cost: 56,604 k€  (supply-demand max gap 0.00 GW)
MILP optimum:            56,604 k€
greedy peaker on/off: [0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0]


On this easy day the merit-order schedule coincides with the MILP optimum: one peaker block,
one start-up, nothing for min up-time to bind on. Greedy breaks when a unit would be switched
on and off several times (two short peaks with a start-up cost each) or when min up-time
forces it to run at a loss — precisely the cases you buy the MILP for. Check greedy against
the exact solution on an instance where you *can* solve exactly before trusting it elsewhere.

In [14]:
# differential_evolution on a small non-convex problem: a 2-parameter 'dispatch rule' tuned by simulation
def rule_cost(theta):
    thresh, share = theta                        # peaker on above 'thresh' GW, taking 'share' of the excess
    p = np.zeros((len(demand), 3))
    for h, d in enumerate(demand):
        peak = np.clip(d - thresh, 0, None) * share
        peak = np.clip(peak, 0, units.p_max["peaker"])
        p[h, 2] = peak if peak >= units.p_min["peaker"] else 0
        rest = d - p[h, 2]
        p[h, 0] = min(units.p_max["nuclear"], rest); rest -= p[h, 0]
        p[h, 1] = rest
        if p[h, 1] > units.p_max["ccgt"] + 1e-9: return 1e9   # infeasible → penalty
    return schedule_cost(p, units)

t0 = time.time()
de = differential_evolution(rule_cost, bounds=[(25, 40), (0.0, 1.0)], seed=0, maxiter=60, tol=1e-6)
print(f"DE: threshold {de.x[0]:.2f} GW, share {de.x[1]:.2f} → cost {de.fun:,.0f} k€ in {time.time()-t0:.1f}s  (MILP {res.fun:,.0f})")

DE: threshold 30.77 GW, share 0.85 → cost 56,883 k€ in 1.4s  (MILP 56,604)


The tuned rule gets close to the exact optimum because the problem is small and the rule
family is sensible. That is the honest use of heuristics: a good rule, checked against the
exact answer where the exact answer is available.

## 8. Timing as the problem grows

`milp` (HiGHS) handles small unit-commitment problems instantly; the horizon and the number
of binaries drive the cost. Measure rather than guess.

In [15]:
rows = []
for days in [1, 2, 4, 7]:
    dem = df["consumption_mwh"].values[:24 * days] / 1000
    c_, cons_, integ_, bnds_, ix_ = build_uc(dem, units)
    t0 = time.time()
    r_ = milp(c_, constraints=cons_, integrality=integ_, bounds=bnds_, options={"time_limit": 60})
    rows.append([days, ix_.n, cons_.A.shape[0], r_.status, time.time() - t0])
pd.DataFrame(rows, columns=["days", "variables", "constraint rows", "status", "seconds"]).round(2)

,days,variables,constraint rows,status,seconds
0,1,216,582,0,0.00
1,2,432,1446,0,0.01
2,4,864,3174,0,0.02
3,7,1512,5766,0,0.06


**Formulation tips**
- Tight `p_max` in `p ≤ p_max·u` (no big-M); bounds in `Bounds`, not as rows.
- Fewer binaries: the start-up indicator `s` can stay continuous — the constraints force it.
- Dense `A` is fine to ~10⁴ rows; beyond that use `scipy.sparse.csr_matrix` for `A`.
- `options={"time_limit": ...}` and check `res.status == 1` for a time-limited, possibly
  sub-optimal answer; `res.mip_gap` tells you how far from proven optimal.
- No warm start in `scipy.optimize.milp`; rolling-horizon problems re-solve from scratch.

## Quick reference

| Task | Tool | Watch out |
|---|---|---|
| LP | `linprog(c, A_ub, b_ub, A_eq, b_eq, bounds, method="highs")` | `res.status`; duals in `res.eqlin.marginals` |
| MILP | `milp(c, constraints=LinearConstraint(A, lb, ub), integrality, bounds=Bounds(lo, hi))` | `integrality` is a float array; bounds separate from `A` |
| Index (g, h) → column | small helper class, asserted on a toy | most MILP bugs are indexing bugs |
| Judge the integer solution | solve the LP relaxation; gap = MILP/LP − 1 | rounding the LP is usually infeasible |
| Pick projects under a budget | knapsack: all-binary `milp` | greedy ratio ≠ optimum |
| Decide under uncertainty | two-stage LP over scenarios; recourse per scenario | evaluate OOS; check `F*` stability across re-draws |
| Risk | add CVaR variables `η, z_s` — still an LP | pick `α`, `λ` deliberately |
| Too slow | greedy → local search → `differential_evolution` | verify against exact on a small instance |